# 06 - Where do the `partial` trajectories go?

A trajectory is `partial` when it reaches the gate day (`DAYS[0]` = 30) but is
deleted before one or more later days - Parcels removed the particle, so the
feature vector is padded with its last known position. After retiring the 126
duplicated stores these are the only losses left: ~13.7k of 15.3M (0.09%),
down from 313k when run 67891's truncated tail was still in.

This notebook asks *what happened to them*: when they were deleted, where they
were when it happened, and whether the loss is a boundary artefact (particle
advected out of the domain) or something physical (beaching on the coast).

1. Select the partials and stream their **full** paths from Zarr.
2. Survival: how long each one lasted, which sampled days it reached.
3. Map the deletion positions.
4. Classify: domain edge vs interior, and where the interior ones cluster.
5. Season / year / run breakdown.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import config as C
import pipeline as P

# --- palette (categorical slots, light surface) ------------------------------
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "0.88"
EDGE_TOL = 1.0        # deg: deleted this close to a domain wall -> "left domain"
N_IO_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", 16))

def style(ax, xlab="", ylab=""):
    ax.set_xlabel(xlab, fontsize=9, color=MUTED)
    ax.set_ylabel(ylab, fontsize=9, color=MUTED)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("0.8")
    ax.tick_params(colors=MUTED, labelsize=8, length=0)
    ax.set_axisbelow(True)
    return ax

print("days:", C.DAYS, "| plot box:", C.DOMAIN, "| stores:", len(C.list_stores()))

days: [30, 50, 100, 150, 180] | plot box: {'lon_min': -95, 'lon_max': 5, 'lat_min': -8, 'lat_max': 27} | stores: 1528


## Select the partials
`status == "partial"` means the gate day was reached but a later day was not.
`early_loss` (never reached the gate) should now be empty - if it is not, the
duplicated stores were not retired.

In [2]:
features = pd.read_parquet(C.FEATURES_FILE)
print(features.status.value_counts().to_string())

part = features[features.status == "partial"].reset_index(drop=True)
n_lab = (features.status != "early_loss").sum()
print(f"\npartial: {len(part):,} of {n_lab:,} labelable ({100*len(part)/n_lab:.3f}%)")

part["run"] = part.store.str.extract(r"Parcels_run_(\d+)_")
print("\nby run:")
print(part.run.value_counts().to_string())

status
complete      15266266
partial          13734
early_loss           0

partial: 13,734 of 15,280,000 labelable (0.090%)

by run:
run
45678    6073
78876    4334
67891    3327


## The real advection domain
`config.DOMAIN` is the **plotting** box, and it is narrower than the box the
particles actually move in - so it cannot be used to decide whether a particle
was deleted at a wall. Measure the true extent from the sampled positions
instead. Partial trajectories are padded with their last known position, so
every deletion point is present in these columns.

In [3]:
_cols = [f"{p}_{d}" for d in C.DAYS for p in ("lat", "lon")]
_la = features[[c for c in _cols if c.startswith("lat")]].to_numpy()
_lo = features[[c for c in _cols if c.startswith("lon")]].to_numpy()
BOX = dict(lat_min=float(np.nanmin(_la)), lat_max=float(np.nanmax(_la)),
           lon_min=float(np.nanmin(_lo)), lon_max=float(np.nanmax(_lo)))
del _la, _lo
for k, v in BOX.items():
    print(f"  {k:8s} {v:8.2f}      (config.DOMAIN: {C.DOMAIN[k]:6.1f})")
print("\n-> the wall test uses BOX, not config.DOMAIN")

  lat_min     -7.48      (config.DOMAIN:   -8.0)
  lat_max     30.02      (config.DOMAIN:   27.0)
  lon_min    -94.96      (config.DOMAIN:  -95.0)
  lon_max      7.88      (config.DOMAIN:    5.0)

-> the wall test uses BOX, not config.DOMAIN


## Stream the full path of every partial
Small subset, so we can afford *all* of them (not a sample). Open each store
once, pull only the rows we need, and record the **last valid observation**:
its time (days since release) and position - that is the deletion event.

In [4]:
# Look the store up by NAME, not by trajectory_id // TRAJ_PER_STORE: retiring
# the duplicated stores renumbered every store_index, so the arithmetic route
# silently points at the wrong file in any features.parquet built before that.
store_paths = {p.stem: p for p in C.list_stores()}
unknown = ~part.store.isin(store_paths)
if unknown.any():
    print(f"WARNING: {unknown.sum():,} partials sit in stores that no longer "
          f"exist (retired duplicates).\n         features.parquet predates the "
          f"cleanup - rerun notebook 01. Ignoring them here.")
    part = part[~unknown].reset_index(drop=True)

part["local"] = part.trajectory_id % C.TRAJ_PER_STORE

def _last_valid(name, grp):
    ds  = xr.open_zarr(store_paths[name])
    loc = grp.local.to_numpy()
    lat  = ds.lat.values[loc]
    lon  = ds.lon.values[loc]
    time = ds.time.values[loc]
    valid = ~np.isnat(time) & ~np.isnan(lon)
    last  = P._last_valid_index(valid)                  # index of final fix
    rows  = np.arange(len(loc))
    rel   = (time[rows, last] - time[:, 0]) / np.timedelta64(1, "D")
    return pd.DataFrame({
        "trajectory_id": grp.trajectory_id.to_numpy(),
        "loss_day": rel.astype(np.float32),
        "loss_lat": lat[rows, last].astype(np.float32),
        "loss_lon": lon[rows, last].astype(np.float32),
        "n_obs":    valid.sum(axis=1).astype(np.int32),
    })

res = Parallel(n_jobs=N_IO_THREADS, backend="threading")(
    delayed(_last_valid)(name, grp) for name, grp in part.groupby("store"))
loss = pd.concat(res, ignore_index=True).merge(
    part[["trajectory_id", "run", "release_time", "release_year",
          "release_month"]], on="trajectory_id")
print(f"streamed {len(loss):,} partial trajectories")
loss.describe()[["loss_day", "loss_lat", "loss_lon"]]

streamed 13,734 partial trajectories


,loss_day,loss_lat,loss_lon
count,13734.000000,13734.000000,13734.000000
mean,166.639603,29.937149,-80.028358
min,136.972229,21.817719,-94.960312
25%,160.055557,29.962133,-80.157665
50%,168.333328,29.965508,-80.031590
75%,174.177086,29.968243,-79.840702
max,178.986115,30.022831,-59.925850
std,8.941788,0.370937,1.437940


## Survival - how far did they get?
`loss_day` is days-since-release at the final fix. The sampled days are marked:
a particle contributes real data up to its loss day and padded (last-known)
positions after it, so the further right it dies, the less its feature vector
is faked.

In [5]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(loss.loss_day, bins=np.arange(0, 185, 2.5), color=BLUE, alpha=.85)
for d in C.DAYS:
    ax.axvline(d, color=MUTED, lw=.8, ls=":", zorder=3)
    ax.annotate(f"day {d}", (d, 1), xycoords=("data", "axes fraction"),
                xytext=(3, -10), textcoords="offset points",
                fontsize=7, color=MUTED, ha="left", va="top")
ax.yaxis.grid(True, color=GRID, lw=.6)
style(ax, "days since release at final fix", "trajectories deleted")
ax.set_title("When the partials are deleted", fontsize=11, color=INK,
             pad=10, loc="left")
fig.tight_layout(); plt.show()

print("how many sampled days each partial actually reached:")
reached = pd.Series({f"reached day {d}": int((loss.loss_day >= d - C.TIME_TOL_DAYS).sum())
                     for d in C.DAYS})
tbl = reached.to_frame("n")
tbl["% of partials"] = (100 * tbl.n / len(loss)).round(1)
tbl["padded days"]   = [sum(1 for x in C.DAYS if x > d) for d in C.DAYS]
print(tbl.to_string())

how many sampled days each partial actually reached:
                     n  % of partials  padded days
reached day 30   13734          100.0            4
reached day 50   13734          100.0            3
reached day 100  13734          100.0            2
reached day 150  13215           96.2            1
reached day 180      0            0.0            0


## Cheap coastline
Same trick as the other notebooks: Natural Earth 110 m as plain (lon, lat)
segments, drawn on ordinary axes.

In [6]:
import cartopy.feature as cfeature
_coast = []
for geom in cfeature.COASTLINE.with_scale("110m").geometries():
    for line in getattr(geom, "geoms", [geom]):
        _coast.append((np.asarray(line.xy[0]), np.asarray(line.xy[1])))

def draw_coast(ax):
    for x, y in _coast:
        ax.plot(x, y, "-", color="0.75", lw=.5, zorder=1)
    ax.set_xlim(BOX["lon_min"] - 2, BOX["lon_max"] + 2)
    ax.set_ylim(BOX["lat_min"] - 2, BOX["lat_max"] + 2)
print("coastline segments:", len(_coast))

coastline segments: 134


## Where are they when they vanish?
Each dot is one deletion, coloured by how long the particle survived. The
dashed rectangle is the model domain - dots sitting on it left through a wall;
dots on the coast are candidate beachings.

In [7]:
fig, ax = plt.subplots(figsize=(11, 5.5))
draw_coast(ax)
sc = ax.scatter(loss.loss_lon, loss.loss_lat, c=loss.loss_day, cmap="viridis",
                s=7, alpha=.6, zorder=3, linewidths=0)
cb = fig.colorbar(sc, ax=ax, pad=.01, fraction=.03)
cb.set_label("days survived", fontsize=8, color=MUTED)
cb.ax.tick_params(colors=MUTED, labelsize=7, length=0)
ax.add_patch(plt.Rectangle(
    (BOX["lon_min"], BOX["lat_min"]),
    BOX["lon_max"] - BOX["lon_min"], BOX["lat_max"] - BOX["lat_min"],
    fill=False, ec=ORANGE, lw=1.2, ls="--", zorder=4))
style(ax, "longitude", "latitude")
ax.set_title(f"Deletion position of all {len(loss):,} partial trajectories",
             fontsize=11, color=INK, pad=10, loc="left")
fig.tight_layout(); plt.show()

## Boundary artefact or something else?
A particle deleted within `EDGE_TOL` of a domain wall was almost certainly
advected out (Parcels deletes out-of-bounds particles). Anything deleted in the
interior needs another explanation - beaching, a land mask cell, or a gap in
the forcing.

In [8]:
d_left   = (loss.loss_lon - BOX["lon_min"]).abs()
d_right  = (BOX["lon_max"] - loss.loss_lon).abs()
d_bottom = (loss.loss_lat - BOX["lat_min"]).abs()
d_top    = (BOX["lat_max"] - loss.loss_lat).abs()
edge = pd.concat([d_left, d_right, d_bottom, d_top], axis=1)
loss["edge_dist"] = edge.min(axis=1)
loss["wall"] = np.where(loss.edge_dist > EDGE_TOL, "interior",
                        np.array(["west", "east", "south", "north"])[edge.values.argmin(1)])

vc = loss.wall.value_counts()
out = vc.to_frame("n"); out["%"] = (100 * out.n / len(loss)).round(1)
print(out.to_string())

fig, ax = plt.subplots(figsize=(7, 3.2))
cols = [ORANGE if w != "interior" else BLUE for w in vc.index]
ax.barh(range(len(vc)), vc.values, color=cols, height=.6)
ax.set_yticks(range(len(vc))); ax.set_yticklabels(vc.index, fontsize=9, color=INK)
ax.invert_yaxis()
ax.xaxis.grid(True, color=GRID, lw=.6)
for i, v in enumerate(vc.values):
    ax.annotate(f"{v:,}", (v, i), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=8, color=MUTED)
style(ax, "trajectories deleted", "")
ax.set_title("Deleted at a domain wall, or in the interior?", fontsize=11,
             color=INK, pad=10, loc="left")
fig.tight_layout(); plt.show()

           n     %
wall              
north  13651  99.4
west      83   0.6


## The interior losses
If these dominate, the deletions are not a boundary artefact. Plot them alone
against the coastline - a tight hug of the shelf means beaching.

In [9]:
inter = loss[loss.wall == "interior"]
print(f"interior losses: {len(inter):,} ({100*len(inter)/len(loss):.1f}% of partials)")

if len(inter):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    draw_coast(ax)
    ax.scatter(inter.loss_lon, inter.loss_lat, s=9, color=BLUE, alpha=.55,
               zorder=3, linewidths=0)
    style(ax, "longitude", "latitude")
    ax.set_title(f"Interior deletions only ({len(inter):,})", fontsize=11,
                 color=INK, pad=10, loc="left")
    fig.tight_layout(); plt.show()

    print("\ninterior losses, survival:")
    print(inter.loss_day.describe().round(1).to_string())

interior losses: 0 (0.0% of partials)


## When does it happen?
Losses concentrated in particular years point at the forcing (a bad chunk);
a seasonal cycle points at circulation. Rates are per-release-batch so the
uneven per-year store counts do not distort the comparison.

In [10]:
tot = features[features.status != "early_loss"].groupby("release_year").size()
by_year = loss.groupby("release_year").size().reindex(tot.index, fill_value=0)
rate = (1e4 * by_year / tot).rename("per 10k released")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].bar(rate.index, rate.values, color=BLUE, width=.7)
axes[0].yaxis.grid(True, color=GRID, lw=.6)
style(axes[0], "release year", "losses per 10k released")
axes[0].set_title("by year", fontsize=10, color=INK, pad=8, loc="left")

tot_m = features[features.status != "early_loss"].groupby("release_month").size()
by_m  = loss.groupby("release_month").size().reindex(tot_m.index, fill_value=0)
axes[1].bar(by_m.index, 1e4 * by_m / tot_m, color=AQUA, width=.7)
axes[1].set_xticks(range(1, 13))
axes[1].yaxis.grid(True, color=GRID, lw=.6)
style(axes[1], "release month", "")
axes[1].set_title("by month", fontsize=10, color=INK, pad=8, loc="left")
fig.tight_layout(); plt.show()

print("loss rate per 10k released, by run:")
tr = features[features.status != "early_loss"].copy()
tr["run"] = tr.store.str.extract(r"Parcels_run_(\d+)_")
print((1e4 * loss.groupby("run").size() / tr.groupby("run").size())
      .round(1).to_string())

loss rate per 10k released, by run:


run
45678     7.6
67891     8.7
78876    12.5


## Verdict
Read off the numbers above:

- **Mostly at a wall** -> boundary artefact. Harmless for clustering: those
  particles left the region of interest, and 180-day paths that exit the domain
  are not physically meaningful anyway. Consider dropping `partial` from
  notebook 03 rather than labelling them with padded positions.
- **Mostly interior, hugging the coast** -> beaching. Worth a Parcels-side fix
  (a coastal-displacement kernel) if you care about the shelf pathways.
- **Concentrated in one year / one run** -> a forcing problem, like the 2003-2005
  window; check that chunk before trusting it.

Whatever the answer, remember the padding: a partial's `lat_d`/`lon_d` for every
day after `loss_day` is its **last known position repeated**, so those feature
vectors sit artificially still. Notebook 1.5 avoids this by fitting on
`complete` only, but notebook 03 labels everything that is not `early_loss`.

In [11]:
OUT = C.DATA_DIR / "partial_losses.parquet"
loss.to_parquet(OUT, index=False)
print("saved", loss.shape, "->", OUT)
print("\ncolumns:", list(loss.columns))

saved (13734, 11) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/partial_losses.parquet

columns: ['trajectory_id', 'loss_day', 'loss_lat', 'loss_lon', 'n_obs', 'run', 'release_time', 'release_year', 'release_month', 'edge_dist', 'wall']
